In [1]:
import scanpy as sc
data_dir = '../../dataset/Marshall2022High_mouse_sampled.h5ad'
adata = sc.read_h5ad(data_dir)

In [2]:
sc.pp.normalize_total(adata)
sc.pp.log1p(adata)

In [3]:
adata.var['gene_id'] = adata.var.index
adata.var.index = adata.var['gene_name']

# 输出 adata.var
adata.var

,feature_is_filtered,feature_reference,feature_biotype,feature_name-0,feature_length-0,feature_type-0,feature_name-1,feature_length-1,feature_type-1,feature_name-2,...,feature_length-5,feature_type-5,feature_name-6,feature_length-6,feature_type-6,feature_name-7,feature_length-7,feature_type-7,gene_name,gene_id
gene_name,,,,,,,,,,,,,,,,,,,,,
Erg28,False,NCBITaxon:10090,gene,Erg28,696,protein_coding,Erg28,696,protein_coding,Erg28,...,696,protein_coding,Erg28,696,protein_coding,Erg28,696,protein_coding,Erg28,ENSMUSG00000021252
0610009B22Rik,False,NCBITaxon:10090,gene,0610009B22Rik,852,protein_coding,0610009B22Rik,852,protein_coding,0610009B22Rik,...,852,protein_coding,0610009B22Rik,852,protein_coding,0610009B22Rik,852,protein_coding,0610009B22Rik,ENSMUSG00000007777
0610009E02Rik,False,NCBITaxon:10090,gene,0610009E02Rik,981,lncRNA,0610009E02Rik,981,lncRNA,0610009E02Rik,...,981,lncRNA,0610009E02Rik,981,lncRNA,0610009E02Rik,981,lncRNA,0610009E02Rik,ENSMUSG00000086714
0610009L18Rik,False,NCBITaxon:10090,gene,0610009L18Rik,639,lncRNA,0610009L18Rik,639,lncRNA,0610009L18Rik,...,639,lncRNA,0610009L18Rik,639,lncRNA,0610009L18Rik,639,lncRNA,0610009L18Rik,ENSMUSG00000043644
Dele1,False,NCBITaxon:10090,gene,Dele1,2394,protein_coding,Dele1,2394,protein_coding,Dele1,...,2394,protein_coding,Dele1,2394,protein_coding,Dele1,2394,protein_coding,Dele1,ENSMUSG00000024442
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
mt-Tm,False,NCBITaxon:10090,gene,mt-Tm,69,Mt_tRNA,mt-Tm,69,Mt_tRNA,mt-Tm,...,69,Mt_tRNA,mt-Tm,69,Mt_tRNA,mt-Tm,69,Mt_tRNA,mt-Tm,ENSMUSG00000064344
mt-Tp,False,NCBITaxon:10090,gene,mt-Tp,67,Mt_tRNA,mt-Tp,67,Mt_tRNA,mt-Tp,...,67,Mt_tRNA,mt-Tp,67,Mt_tRNA,mt-Tp,67,Mt_tRNA,mt-Tp,ENSMUSG00000064372
mt-Tq,False,NCBITaxon:10090,gene,mt-Tq,71,Mt_tRNA,mt-Tq,71,Mt_tRNA,mt-Tq,...,71,Mt_tRNA,mt-Tq,71,Mt_tRNA,mt-Tq,71,Mt_tRNA,mt-Tq,ENSMUSG00000064343


In [4]:
import scanpy as sc
import torch
import lightning.pytorch as pl
from self_supervision.models.lightning_modules.cellnet_autoencoder import MLPAutoEncoder
from self_supervision.estimator.cellnet import EstimatorAutoEncoder

# 设置你的 .ckpt 文件路径
ckpt_path = "../../sc_pretrained/Pretrained Models/RandomMask.ckpt"

# 模型参数
units_encoder = [512, 512, 256, 256, 64]
units_decoder = [256, 256, 512, 512]

# 初始化 EstimatorAutoEncoder 实例
estim = EstimatorAutoEncoder(data_path=None)  # 如果没有实际数据路径，可以设置为None

# 加载预训练模型
estim.model = MLPAutoEncoder.load_from_checkpoint(
    ckpt_path,
    gene_dim=19331,  # 根据你的数据调整
    batch_size=128,  # 根据你的需要调整
    units_encoder=units_encoder, 
    units_decoder=units_decoder,
    masking_strategy="random",  # 假设模型使用了随机掩码
    masking_rate=0.5,  # 根据需要调整
)

# 使用 GPU 进行评估（如果可用）
estim.trainer = pl.Trainer(accelerator="gpu", devices=1 if torch.cuda.is_available() else None)
estim.model

/home/hanchuangyi/miniconda3/envs/ssl/lib/python3.10/site-packages/merlin/dtypes/mappings/tf.py:52: UserWarning: Tensorflow dtype mappings did not load successfully due to an error: No module named 'tensorflow'
  warn(f"Tensorflow dtype mappings did not load successfully due to an error: {exc.msg}")
/home/hanchuangyi/miniconda3/envs/ssl/lib/python3.10/site-packages/merlin/dtypes/mappings/triton.py:53: UserWarning: Triton dtype mappings did not load successfully due to an error: No module named 'tritonclient'
  warn(f"Triton dtype mappings did not load successfully due to an error: {exc.msg}")


GPU available: True (cuda), used: True


TPU available: False, using: 0 TPU cores


HPU available: False, using: 0 HPUs


MLPAutoEncoder(
  (train_metrics): MetricCollection(
    (explained_var_uniform): ExplainedVariance()
    (explained_var_weighted): ExplainedVariance()
    (mse): MeanSquaredError(),
    prefix=train_
  )
  (val_metrics): MetricCollection(
    (explained_var_uniform): ExplainedVariance()
    (explained_var_weighted): ExplainedVariance()
    (mse): MeanSquaredError(),
    prefix=val_
  )
  (test_metrics): MetricCollection(
    (explained_var_uniform): ExplainedVariance()
    (explained_var_weighted): ExplainedVariance()
    (mse): MeanSquaredError(),
    prefix=test_
  )
  (encoder): MLP(
    (0): Linear(in_features=19331, out_features=512, bias=True)
    (1): SELU()
    (2): Dropout(p=0.1, inplace=False)
    (3): Linear(in_features=512, out_features=512, bias=True)
    (4): SELU()
    (5): Dropout(p=0.1, inplace=False)
    (6): Linear(in_features=512, out_features=256, bias=True)
    (7): SELU()
    (8): Dropout(p=0.1, inplace=False)
    (9): Linear(in_features=256, out_features=256, b

In [5]:
import pandas as pd
var_df = pd.read_parquet('../../sc_pretrained/var.parquet')
var_df

,feature_id,feature_name
0,ENSG00000186092,OR4F5
1,ENSG00000284733,OR4F29
2,ENSG00000284662,OR4F16
3,ENSG00000187634,SAMD11
4,ENSG00000188976,NOC2L
...,...,...
19326,ENSG00000288702,UGT1A3
19327,ENSG00000288705,UGT1A5
19328,ENSG00000182484,WASH6P
19329,ENSG00000288622,PDCD6-AHRR


In [6]:
all_genes = var_df['feature_name'].tolist()
all_genes

['OR4F5',
 'OR4F29',
 'OR4F16',
 'SAMD11',
 'NOC2L',
 'KLHL17',
 'PLEKHN1',
 'PERM1',
 'HES4',
 'ISG15',
 'AGRN',
 'RNF223',
 'C1orf159',
 'TTLL10',
 'TNFRSF18',
 'TNFRSF4',
 'SDF4',
 'B3GALT6',
 'C1QTNF12',
 'UBE2J2',
 'SCNN1D',
 'ACAP3',
 'PUSL1',
 'INTS11',
 'CPTP',
 'TAS1R3',
 'DVL1',
 'MXRA8',
 'AURKAIP1',
 'CCNL2',
 'MRPL20',
 'ANKRD65',
 'TMEM88B',
 'VWA1',
 'ATAD3C',
 'ATAD3B',
 'ATAD3A',
 'TMEM240',
 'SSU72',
 'FNDC10',
 'MIB2',
 'MMP23B',
 'CDK11B',
 'SLC35E2B',
 'CDK11A',
 'NADK',
 'GNB1',
 'CALML6',
 'TMEM52',
 'CFAP74',
 'GABRD',
 'PRKCZ',
 'FAAP20',
 'SKI',
 'MORN1',
 'RER1',
 'PEX10',
 'PLCH2',
 'PANK4',
 'HES5',
 'TNFRSF14',
 'PRXL2B',
 'MMEL1',
 'TTC34',
 'ACTRT2',
 'PRDM16',
 'ARHGEF16',
 'MEGF6',
 'TPRG1L',
 'WRAP73',
 'TP73',
 'CCDC27',
 'SMIM1',
 'LRRC47',
 'CEP104',
 'DFFB',
 'C1orf174',
 'AJAP1',
 'NPHP4',
 'KCNAB2',
 'CHD5',
 'RPL22',
 'RNF207',
 'ICMT',
 'HES3',
 'GPR153',
 'ACOT7',
 'HES2',
 'ESPN',
 'TNFRSF25',
 'PLEKHG5',
 'NOL9',
 'TAS1R1',
 'ZBTB48',
 'KLH

In [7]:
adata.var['gene_name']=adata.var.index
adata.var['gene_name']

gene_name
Erg28                    Erg28
0610009B22Rik    0610009B22Rik
0610009E02Rik    0610009E02Rik
0610009L18Rik    0610009L18Rik
Dele1                    Dele1
                     ...      
mt-Tm                    mt-Tm
mt-Tp                    mt-Tp
mt-Tq                    mt-Tq
mt-Tt                    mt-Tt
mt-Tv                    mt-Tv
Name: gene_name, Length: 16661, dtype: category
Categories (20499, object): ['0610009B22Rik', '0610009E02Rik', '0610009L18Rik', '0610010K14Rik', ..., 'mt-Tq', 'mt-Ts2', 'mt-Tt', 'mt-Tv']

In [8]:
import numpy as np
# 初始化一个新的数据矩阵，形状为 (adata.X.shape[0], len(all_genes))，填充为零
new_data = np.zeros((adata.X.shape[0], len(all_genes)), dtype=np.float32)

In [9]:
existing_genes = adata.var['gene_name']
existing_genes

gene_name
Erg28                    Erg28
0610009B22Rik    0610009B22Rik
0610009E02Rik    0610009E02Rik
0610009L18Rik    0610009L18Rik
Dele1                    Dele1
                     ...      
mt-Tm                    mt-Tm
mt-Tp                    mt-Tp
mt-Tq                    mt-Tq
mt-Tt                    mt-Tt
mt-Tv                    mt-Tv
Name: gene_name, Length: 16661, dtype: category
Categories (20499, object): ['0610009B22Rik', '0610009E02Rik', '0610009L18Rik', '0610010K14Rik', ..., 'mt-Tq', 'mt-Ts2', 'mt-Tt', 'mt-Tv']

In [10]:
# 将所有基因名称转换为小写
all_genes_lower = [gene.lower() for gene in all_genes]
adata_genes_lower = [gene.lower() for gene in existing_genes]

# 将两个列表转换为集合
all_genes_set = set(all_genes_lower)
adata_genes_set = set(adata_genes_lower)

# 计算交集
matching_genes = all_genes_set.intersection(adata_genes_set)
matching_count = len(matching_genes)
# 计算不匹配的基因
non_matching_genes = adata_genes_set - matching_genes
non_matching_count = len(non_matching_genes)


# 输出结果
print(f"匹配的基因数量: {matching_count}")
print(f"匹配的基因列表: {matching_genes}")
non_matching_genes


匹配的基因数量: 13074
匹配的基因列表: {'dhx33', 'snrpa', 'nrap', 'fbxl7', 'setdb2', 'pxmp2', 'faf1', 'nedd1', 'ica1', 'ppt2', 'rbm14', 'actr10', 'rimkla', 'p2rx1', 'chrna4', 'slc7a7', 'mmgt1', 'c3ar1', 'raph1', 'alpl', 'lrrc75a', 'msh4', 'ik', 'morf4l1', 'smchd1', 'prdx1', 'cbx5', 'tmigd1', 'smim10l2a', 'sppl3', 'neil3', 'rps13', 'pank1', 'fbxo16', 'herc6', 'kidins220', 'mx1', 'smad4', 'ptprz1', 'angel2', 'acsf2', 'ptpn3', 'rab11b', 'rnd2', 'clasp2', 'ppp1r16a', 'txlng', 'gng7', 'pgm2l1', 'xpo5', 'mmp15', 'camkk2', 'csk', 'hddc2', 'armc9', 'pcgf3', 'nmd3', 'cks2', 'macrod2', 'knl1', 'gabra2', 'rsph1', 'mctp1', 'mxi1', 'med1', 'larp4', 'med31', 'col4a6', 'atl3', 'cbx3', 'focad', 'pak6', 'rgs2', 'slc30a6', 'ltc4s', 'pkd1l1', 'nmnat1', 'prdm2', 'procr', 'pwp1', 'ctcf', 'dlec1', 'erbb3', 'casr', 'fbxo42', 'pgm5', 'med17', 'coq5', 'garem1', 'samd5', 'itgb1bp2', 'slc25a5', 'ube2u', 'aph1a', 'doc2b', 'wdr82', 'rpl38', 'atp6v1g1', 'fos', 'zfand6', 'fbxl6', 'tspear', 'hectd1', 'mrps31', 'slc20a2', 'foxs1', '

{'b130046b21rik',
 'gm43274',
 'vmn2r103',
 'b630019k06rik',
 'apol7e',
 'trp53i13',
 'gm43136',
 '2010003k11rik',
 '4930453n24rik',
 'gm28863',
 'uqcc6',
 'gm43859',
 'gm27030',
 '4930520o04rik',
 'gm43482',
 'gm38336',
 '3110082i17rik',
 'gm10451',
 '1700067k01rik',
 'gm9755',
 'gm14399',
 '3300002i08rik',
 'gm4208',
 'gramd2',
 'a930001c03rik',
 'zfp235',
 'bc049762',
 'trp53inp2',
 '6720427i07rik',
 'ldc1',
 '3222401l13rik',
 'zfta',
 'gm20404',
 'h2bc27',
 '2610206c17rik',
 'ube2d2a',
 'a230065n10rik',
 'a230072e10rik',
 'slc7a12',
 '4930481b07rik',
 '3010003l21rik',
 'cyp2j8',
 'slfn9',
 'vmn1r4',
 'mir670hg',
 'gm4221',
 'notumos',
 'ifi207',
 'vdac3-ps1',
 'gm13427',
 'gm20422',
 'trac',
 'gm2164',
 'gm21972',
 '4833439l19rik',
 'gm9725',
 'gm11734',
 'slc22a19',
 'gm42447',
 'gm43336',
 'd630033o11rik',
 'zfp119b',
 'ifi204',
 'c630043f03rik',
 '4930594m22rik',
 'zfp354b',
 'dynlt1a',
 'gm17634',
 'gm20442',
 'gm6871',
 'tha1',
 'gm43618',
 'h3f3aos',
 'serpinb6a',
 'e130304i0

In [11]:
gene_to_index = {gene: idx for idx, gene in enumerate(all_genes_lower)}
gene_to_index

{'or4f5': 0,
 'or4f29': 1,
 'or4f16': 2,
 'samd11': 3,
 'noc2l': 4,
 'klhl17': 5,
 'plekhn1': 6,
 'perm1': 7,
 'hes4': 8,
 'isg15': 9,
 'agrn': 10,
 'rnf223': 11,
 'c1orf159': 12,
 'ttll10': 13,
 'tnfrsf18': 14,
 'tnfrsf4': 15,
 'sdf4': 16,
 'b3galt6': 17,
 'c1qtnf12': 18,
 'ube2j2': 19,
 'scnn1d': 20,
 'acap3': 21,
 'pusl1': 22,
 'ints11': 23,
 'cptp': 24,
 'tas1r3': 25,
 'dvl1': 26,
 'mxra8': 27,
 'aurkaip1': 28,
 'ccnl2': 29,
 'mrpl20': 30,
 'ankrd65': 31,
 'tmem88b': 32,
 'vwa1': 33,
 'atad3c': 34,
 'atad3b': 35,
 'atad3a': 36,
 'tmem240': 37,
 'ssu72': 38,
 'fndc10': 39,
 'mib2': 40,
 'mmp23b': 41,
 'cdk11b': 42,
 'slc35e2b': 43,
 'cdk11a': 44,
 'nadk': 45,
 'gnb1': 46,
 'calml6': 47,
 'tmem52': 48,
 'cfap74': 49,
 'gabrd': 50,
 'prkcz': 51,
 'faap20': 52,
 'ski': 53,
 'morn1': 54,
 'rer1': 55,
 'pex10': 56,
 'plch2': 57,
 'pank4': 58,
 'hes5': 59,
 'tnfrsf14': 60,
 'prxl2b': 61,
 'mmel1': 62,
 'ttc34': 63,
 'actrt2': 64,
 'prdm16': 65,
 'arhgef16': 66,
 'megf6': 67,
 'tprg1l': 68

In [12]:
only_in_all_genes = all_genes_set - adata_genes_set

only_in_adata_genes = adata_genes_set - all_genes_set

# 输出结果
print(f"仅在 all_genes 中存在的基因数量: {len(only_in_all_genes)}")
print(f"仅在 all_genes 中存在的基因: {only_in_all_genes}")

print(f"仅在 adata_genes 中存在的基因数量: {len(only_in_adata_genes)}")
print(f"仅在 adata_genes 中存在的基因: {only_in_adata_genes}")


仅在 all_genes 中存在的基因数量: 6257
仅在 all_genes 中存在的基因: {'lilra2', 'or4f15', 'lrrc25', 'znf346', 'hnrnpcl4', 'srsf12', 'znf248', 'unc5a', 'znf33a', 'exd3', 'ccni2', 'gpr50', 'h2al1rp', 'kctd19', 'znf589', 'znf791', 'rbp5', 'krtap24-1', 'gsx1', 'prb4', 'mtrnr2l8', 'elovl4', 'oxtr', 'magea8', 'tnfsf11', 'lce1d', 'or13j1', 'higd2b', 'spata31a7', 'samd11', 'rhoxf2b', 'chrnd', 'gdf3', 'znf740', 'rnase3', 'krtap10-3', 'clca2', 'plekhg7', 'znf644', 'il31', 'dkkl1', 'eqtn', 'spag11b', 'txnrd2', 'vhll', 'c16orf54', 'znf74', 'chrna3', 'pspn', 'znf17', 'znf195', 'prr35', 'mchr2', 'or12d3', 'ccdc102b', 'mrps23', 'fam155b', 'ascl1', 'hsd11b1l', 'kcnh2', 'f2', 'serpina4', 'znf266', 'higd1c', 'sult1a2', 'gdnf', 'znf816-znf321p', 'nmrk2', 'ankrd30b', 'prr33', 'col17a1', 'zbtb47', 'recql4', 'ca11', 'kiaa1191', 'msi1', 'ttc24', 'znf891', 'got1l1', 'actrt3', 'gngt1', 'znf714', 'asb10', 'krt75', 'tubb', 'fgf5', 'rars1', 'raet1g', 'casp5', 'bcan', 'ces5a', 'duoxa2', 'tns4', 'ceacam20', 'defb125', 'icam3', 'iars1'

In [13]:
import numpy as np
from scipy.sparse import csr_matrix

# Initialize a mapping from gene names in adata to their column indices
adata_gene_to_index = {gene: idx for idx, gene in enumerate(adata_genes_lower)}

# Create an array to map adata.X column indices to new_data column indices
adata_to_new_data_indices = -1 * np.ones(adata.X.shape[1], dtype=int)
for idx, gene in enumerate(adata_genes_lower):
    if gene in gene_to_index:
        adata_to_new_data_indices[idx] = gene_to_index[gene]

In [14]:

# Extract data from adata.X without converting it to a dense array
data = adata.X.data
indices = adata.X.indices
indptr = adata.X.indptr

# Map the column indices to the new indices in new_data
mapped_indices = adata_to_new_data_indices[indices]

# Filter out entries where the mapping is invalid (-1)
valid_entries = mapped_indices != -1
new_data_values = data[valid_entries]
new_data_indices = mapped_indices[valid_entries]

# Build the new indptr array for the new_data matrix
new_indptr = np.zeros(adata.X.shape[0] + 1, dtype=int)

In [15]:
for i in range(adata.X.shape[0]):
    row_start = indptr[i]
    row_end = indptr[i + 1]
    valid_count = np.sum(valid_entries[row_start:row_end])
    new_indptr[i + 1] = new_indptr[i] + valid_count

In [16]:
# Construct the new_data sparse matrix
new_data = csr_matrix(
    (new_data_values, new_data_indices, new_indptr),
    shape=(adata.X.shape[0], len(all_genes)),
    dtype=np.float32
)


In [17]:
new_data = new_data.toarray()

In [18]:
from sklearn.metrics import accuracy_score, f1_score
from sklearn.preprocessing import LabelEncoder
import numpy as np
from sklearn.model_selection import train_test_split


label_encoder = LabelEncoder()
labels_encoded = label_encoder.fit_transform(adata.obs['cell_type'])  # 预先编码标签


random_seed = 42
X_train_val, X_test, y_train_val, y_test = train_test_split(
    new_data, labels_encoded, test_size=0.15, random_state=random_seed)


X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.1765, random_state=random_seed)  # 0.1765 是为了让验证集占 15%

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

estim.model.eval()
with torch.no_grad():
    X_train_tensor = torch.tensor(X_train).float().to(device)
    X_test_tensor = torch.tensor(X_test).float().to(device)
    train_embeddings = estim.model.encoder(X_train_tensor).detach().cpu().numpy()
    test_embeddings = estim.model.encoder(X_test_tensor).detach().cpu().numpy()


cuda


In [19]:
import pandas as pd
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report

    

    # 初始化和训练KNN分类器
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(train_embeddings, y_train)
    
    # 模型预测
predictions = knn.predict(test_embeddings)

    # 计算准确率和 F1 分数
accuracy = accuracy_score(y_test, predictions)
print(f"KNN Accuracy on Test Data: {accuracy}")
f1 = f1_score(y_test, predictions, average='weighted')
print(f"Weighted F1 Score: {f1}")
    
macro_f1 = f1_score(y_test, predictions, average='macro')
print(f'Macro F1 Score: {macro_f1}')

    # 计算随机猜测的准确率
class_probabilities = np.bincount(y_test) / len(y_test)
random_accuracy = np.sum(class_probabilities ** 2)
print(f"Random Guess Accuracy: {random_accuracy}")

    # 生成分类报告
report = classification_report(y_test, predictions, target_names=label_encoder.classes_)
print(report)

KNN Accuracy on Test Data: 0.46086991510079267
Weighted F1 Score: 0.4311292862890625
Macro F1 Score: 0.17269234444250775
Random Guess Accuracy: 0.2526864384339867
                                                           precision    recall  f1-score   support

                          blood vessel smooth muscle cell       0.14      0.09      0.11       483
                                         endothelial cell       0.40      0.52      0.46      7705
                 kidney collecting duct intercalated cell       0.10      0.01      0.03        68
                    kidney collecting duct principal cell       0.10      0.02      0.03       376
          kidney distal convoluted tubule epithelial cell       0.58      0.18      0.27       509
                                     kidney granular cell       0.00      0.00      0.00        21
                           kidney interstitial fibroblast       0.28      0.04      0.07       270
kidney loop of Henle thick ascending limb ep

/home/hanchuangyi/miniconda3/envs/ssl/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/hanchuangyi/miniconda3/envs/ssl/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/hanchuangyi/miniconda3/envs/ssl/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.

In [20]:

import pandas as pd
import os
import re

# 当前 Notebook 文件名
notebook_name = "slide_seq_mouse_kidney_random_mask_zero_shot_42.ipynb"

# 初始化需要打印的值
init_train_loss = train_losses[0] if 'train_losses' in globals() else None
init_val_loss = val_losses[0] if 'val_losses' in globals() else None
converged_epoch = len(train_losses) - patience if 'train_losses' in globals() else None
converged_val_loss = best_val_loss if 'best_val_loss' in globals() else None

# 打印所有所需的指标
print("Metrics Summary:")
if 'train_losses' in globals():
    print(f"init_train_loss\tinit_val_loss\tconverged_epoch\tconverged_val_loss\tmacro_f1\tweighted_f1\tmicor_f1")
    print(f"{init_train_loss:.3f}\t{init_val_loss:.3f}\t{converged_epoch}\t{converged_val_loss:.3f}\t{macro_f1:.3f}\t{f1:.3f}\t{accuracy:.3f}")
else:
    print(f"macro_f1\tweighted_f1\tmicor_f1")
    print(f"{macro_f1:.3f}\t{f1:.3f}\t{accuracy:.3f}")

# 保存结果到 CSV 文件
output_data = {
    'dataset_split_random_seed': [int(random_seed)],
    'dataset': ['slide_seq_mouse_kidney'],
    'method': [re.search(r'kidney_(.*?)_\d+', notebook_name).group(1)],
    'init_train_loss': [init_train_loss if init_train_loss is not None else ''],
    'init_val_loss': [init_val_loss if init_val_loss is not None else ''],
    'converged_epoch': [converged_epoch if converged_epoch is not None else ''],
    'converged_val_loss': [converged_val_loss if converged_val_loss is not None else ''],
    'macro_f1': [macro_f1],
    'weighted_f1': [f1],
    'micor_f1': [accuracy]
}
output_df = pd.DataFrame(output_data)

# 保存到当前目录下名为 results 的文件夹中
if not os.path.exists('results'):
    os.makedirs('results')

csv_filename = f"results/{os.path.splitext(notebook_name)[0]}_results.csv"
output_df.to_csv(csv_filename, index=False)


Metrics Summary:
macro_f1	weighted_f1	micor_f1
0.173	0.431	0.461
